# Livestock Sentinel AI: Sprint 2 real data download

This notebook downloads two **real, public** datasets for the 28 sub-counties in the prototype:

1. **Sub-county boundaries** from geoBoundaries (open licence), so the map shows real shapes.
2. **Weather 2013 to 2025**: daily rainfall and evapotranspiration (ERA5 reanalysis via the free Open-Meteo archive API). 2013-2022 is used as the "normal" baseline; 2023-2025 matches the prototype's simulation period.

**How to use:** click **Runtime → Run all**. It takes about 3-5 minutes. At the end, two files download to your computer: `boundaries.geojson` and `weather_weekly.csv`.

In [ ]:
import requests, time, json
import pandas as pd, geopandas as gpd

AREAS_URL = "https://raw.githubusercontent.com/joshuamutahi9/livestock-sentinel/main/data/areas.csv"
areas = pd.read_csv(AREAS_URL)
print(len(areas), "sub-counties loaded")
areas.head()

## 1. Sub-county boundaries

In [ ]:
meta = requests.get("https://www.geoboundaries.org/api/current/gbOpen/KEN/ADM2/", timeout=60).json()
adm2 = gpd.read_file(meta["simplifiedGeometryGeoJSON"]).to_crs(4326)
print(len(adm2), "Kenya ADM2 units downloaded")

pts = gpd.GeoDataFrame(areas, geometry=gpd.points_from_xy(areas.lon, areas.lat), crs=4326)
joined = gpd.sjoin(pts, adm2[["shapeName", "geometry"]], how="left", predicate="within")

# points that fell outside every polygon: take the nearest one
missing = joined["shapeName"].isna()
if missing.any():
    near = gpd.sjoin_nearest(pts[missing.values].to_crs(32737), adm2[["shapeName","geometry"]].to_crs(32737), how="left")
    joined.loc[missing, "shapeName"] = near["shapeName"].values

check = joined[["area_id","county","sub_county","shapeName"]]
dupes = check[check.shapeName.duplicated(keep=False)]
print("\nMatched boundary for each sub-county:")
print(check.to_string(index=False))
print("\nWARNING - these share one polygon:" if len(dupes) else "\nAll sub-counties matched to separate polygons.")
if len(dupes): print(dupes.to_string(index=False))

In [ ]:
out = adm2.merge(check.drop_duplicates("shapeName"), on="shapeName")[["area_id","county","sub_county","shapeName","geometry"]]
out["geometry"] = out.geometry.simplify(0.005)
out.to_file("boundaries.geojson", driver="GeoJSON")
print(len(out), "polygons saved")
out.plot(figsize=(6,6), edgecolor="white")

## 2. Weather (rainfall and evapotranspiration)

In [ ]:
frames = []
for r in areas.itertuples():
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={r.lat}&longitude={r.lon}&start_date=2013-01-01&end_date=2025-12-31"
           "&daily=precipitation_sum,et0_fao_evapotranspiration&timezone=Africa%2FNairobi")
    for attempt in range(4):
        resp = requests.get(url, timeout=120)
        if resp.status_code == 200: break
        time.sleep(10 * (attempt + 1))
    d = resp.json()["daily"]
    frames.append(pd.DataFrame({"area_id": r.area_id, "date": pd.to_datetime(d["time"]),
                                "precip_mm": d["precipitation_sum"], "et0_mm": d["et0_fao_evapotranspiration"]}))
    print(r.sub_county, "done")
    time.sleep(1.5)
daily = pd.concat(frames)
print(len(daily), "daily rows")

In [ ]:
START = pd.Timestamp("2023-01-02")
daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
wk = daily.groupby(["area_id","week_start"], as_index=False)[["precip_mm","et0_mm"]].sum()
wk["woy"] = wk.week_start.dt.isocalendar().week.clip(upper=52).astype(int)

base = wk[(wk.week_start >= "2013-01-07") & (wk.week_start < "2023-01-01")]
clim = base.groupby(["area_id","woy"], as_index=False).agg(precip_clim_mm=("precip_mm","mean"), et0_clim_mm=("et0_mm","mean"))

sim = wk[(wk.week_start >= START) & (wk.week_start < START + pd.Timedelta(weeks=156))].merge(clim, on=["area_id","woy"])
sim["week"] = ((sim.week_start - START).dt.days // 7).astype(int)
sim = sim.sort_values(["area_id","week"])[["area_id","week","week_start","precip_mm","et0_mm","precip_clim_mm","et0_clim_mm"]]
sim.to_csv("weather_weekly.csv", index=False)
print(sim.shape, "- expected (4368, 7)")
sim.groupby("area_id").precip_mm.sum().div(3).round().sort_values().to_frame("avg annual rainfall mm")

## 3. Download the files
Both files will download to your computer. Send them to Claude in the chat.

In [ ]:
from google.colab import files
files.download("boundaries.geojson")
files.download("weather_weekly.csv")